In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [3]:
DATA_FILE = "/ObesityDataSet_raw_and_data_sinthetic.csv"
TARGET = "NObeyesdad"

DROP_COLS = ["Height", "Weight"]
SEED = 42

MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)

X = df.drop(columns=[TARGET] + DROP_COLS)
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
#2025NS04055

Shape: (2111, 17)


In [4]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print("Features:", len(X.columns), "| categorical:", cat_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
#2025NS04055

Features: 14 | categorical: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


In [5]:
def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    #2025NS04055

In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, random_state=SEED),
    "kNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=SEED),
}
#2025NS04055

In [7]:
rows = []
for name, clf in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)
    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba, multi_class="ovr", average="weighted"),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, pred, average="weighted"),
        "F1": f1_score(y_test, pred, average="weighted"),
        "MCC": matthews_corrcoef(y_test, pred),
    })
    fname = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipe, MODEL_DIR / fname)
    print("Saved", fname)

metrics = pd.DataFrame(rows).round(4)
metrics.to_csv(MODEL_DIR / "metrics.csv", index=False)
print(metrics.to_string(index=False))

joblib.dump(le, MODEL_DIR / "label_encoder.joblib")
joblib.dump(list(X.columns), MODEL_DIR / "feature_columns.joblib")
#2025NS04055

Saved logistic_regression.joblib
Saved decision_tree.joblib
Saved knn.joblib
Saved naive_bayes.joblib
Saved support_vector_machine.joblib
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.6241 0.8860     0.6087  0.6241 0.5925 0.5676
         Decision Tree    0.7210 0.8558     0.7207  0.7210 0.7197 0.6745
                   kNN    0.7139 0.9359     0.7204  0.7139 0.6932 0.6717
           Naive Bayes    0.4988 0.8166     0.5360  0.4988 0.4479 0.4387
Support Vector Machine    0.7589 0.9507     0.7607  0.7589 0.7561 0.7192


['model/feature_columns.joblib']

In [9]:
test_df = X_test.copy()
test_df[TARGET] = le.inverse_transform(y_test)
test_df.to_csv("test_data.csv", index=False)
print("test_data.csv rows:", len(test_df))
#2025NS04055

test_data.csv rows: 423
